# CPG-RL 訓練 Notebook（Go2 · MJX · Colab GPU）

把 `task4/cpg_rl_implementation_guide.md` 的訓練端（第 1~6 步）做成可執行 notebook。
**由上往下逐格執行。** 第一次請務必先跑到「Smoke test」那格確認環境能通，再開大規模訓練。

流程：裝套件 → clone Go2 模型 → JAX 版 CPG → 取 IK 常數 → 建 MJX 環境 → smoke test → Brax PPO 訓練 → 看影片 → 存權重下載。

> ⚠️ 重要心理準備：MJX / brax / jax 版本會漂移，本 notebook 是「盡量能直接跑」的最佳版本，但**若某格報 API 錯誤，多半是版本差異**，照錯誤訊息微調即可（大多是函數改名或參數名變動）。Smoke test 就是用來早點、便宜地抓這類錯。

> 這份用的是 **`scene_mjx.xml`**（Go2 的 MJX 專用模型），致動器是**位置伺服**（`ctrl = 關節目標角度`，內建 PD kp=50/kd=0.5），和 task3 的「力矩+軟體PD」不同——所以這裡直接把 CPG 算出的關節角度餵給 `ctrl`。

## 第 1 步：開 GPU + 裝套件

先把 Colab 執行階段改成 GPU：**執行階段 → 變更執行階段類型 → 硬體加速器：GPU（T4 即可）**。然後跑下面兩格。

In [ ]:
# 安裝（約 1~2 分鐘）
!pip install -q mujoco mujoco-mjx brax mediapy
print("done")

In [ ]:
import jax
print("JAX version:", jax.__version__)
print("devices:", jax.devices())   # 必須看到 cuda，不是 cpu；若是 cpu 回去把加速器改成 GPU 再重開執行階段

## 第 1.5 步：取得 Go2 模型

從 MuJoCo Menagerie 抓 Go2（含 MJX 版模型與網格）。

In [ ]:
import os, subprocess
if not os.path.exists("mujoco_menagerie"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/google-deepmind/mujoco_menagerie.git"], check=True)
SCENE = "mujoco_menagerie/unitree_go2/scene_mjx.xml"
print("model exists:", os.path.exists(SCENE))

## 第 2 步：JAX 版 CPG（核心，唯一自己寫的演算法）

把「固定相位時鐘」換成 **Hopf 振盪器**：RL 每一步輸出每腿振幅目標 `mu`（4）與共用頻率 `omega`（1）＝**5 維動作**。
- `cpg_step`：積分振盪器（內部再切 4 個微步，避免 Euler 在 dt=0.02 下不穩）。
- `action_to_cpg_cmd`：把策略輸出 [-1,1]^5 轉成 (mu, omega)。
- `cpg_to_joint_targets`：振盪器狀態 → 12 個關節目標角度（用第 3 步的 IK 常數）。

In [ ]:
import jax.numpy as jnp

A_CONV = 50.0                       # 振幅收斂係數 a
MU_MIN, MU_MAX = 0.0, 1.0           # 振幅目標（0=原地踏、1=滿步幅）
OMEGA_MIN, OMEGA_MAX = 0.5, 3.5     # 頻率 (Hz)
STRIDE = 0.32                       # 步幅 (m) ← 對齊 task3
LIFT   = 0.12                       # 抬腳 (m) ← 對齊 task3
# trot 相位偏移 FL,FR,RL,RR（對角同相）
PHASE_OFFSET = jnp.array([0.0, jnp.pi, jnp.pi, 0.0])
N_CPG_SUB = 4                       # CPG 積分微步數


def cpg_init():
    return {"r": jnp.zeros(4), "r_dot": jnp.zeros(4), "phi": jnp.array(0.0)}


def cpg_step(cpg, mu, omega, dt):
    r, r_dot, phi = cpg["r"], cpg["r_dot"], cpg["phi"]
    h = dt / N_CPG_SUB
    for _ in range(N_CPG_SUB):       # 展開的微步 Euler 積分
        r_ddot = A_CONV * (A_CONV / 4.0 * (mu - r) - r_dot)
        r_dot = r_dot + r_ddot * h
        r = r + r_dot * h
        phi = phi + 2.0 * jnp.pi * omega * h
    phi = jnp.mod(phi, 2.0 * jnp.pi)
    return {"r": r, "r_dot": r_dot, "phi": phi}


def action_to_cpg_cmd(action):
    a = jnp.tanh(action)             # 夾到 [-1,1]
    mu = (a[:4] + 1.0) / 2.0 * (MU_MAX - MU_MIN) + MU_MIN
    omega = (a[4] + 1.0) / 2.0 * (OMEGA_MAX - OMEGA_MIN) + OMEGA_MIN
    return mu, omega


def cpg_to_joint_targets(cpg, f0, jinv, center):
    theta = cpg["phi"] + PHASE_OFFSET
    amp = jnp.clip(cpg["r"], 0.0, 1.0)
    # 站立相(sin<=0)腳掌前->後推身體前進；擺動相(sin>0)後->前回擺
    dx = -STRIDE * amp * jnp.cos(theta)
    dz = jnp.maximum(0.0, LIFT * amp * jnp.sin(theta))
    foot = jnp.stack([center[0] + dx, center[1] + dz], axis=-1)   # (4,2)
    dq = (foot - f0) @ jinv.T                                     # (4,2)->(dth,dca)
    q = jnp.zeros((4, 3))
    q = q.at[:, 0].set(0.0)                # hip 固定 0（純矢狀面 trot）
    q = q.at[:, 1].set(0.9 + dq[:, 0])     # thigh = home 0.9 + dth
    q = q.at[:, 2].set(-1.8 + dq[:, 1])    # calf  = home -1.8 + dca
    return q.reshape(12)

# 快速自測
_c = cpg_init()
_c = cpg_step(_c, jnp.ones(4) * 0.5, 2.0, 0.02)
print("cpg ok, phi=", float(_c["phi"]))

## 第 3 步：算出 IK 常數（沿用你 task3 的數值 Jacobian 做法）

Go2 在 home 姿態附近，腳掌 (x,z) 對 (thigh,calf) 的 Jacobian 是常數。用 CPU 模型算一次，之後當 JAX 常數。
`center=[f0_x, z0]`，`z0` 是站立時腳掌下壓量（讓腿載重），沿用 task3 的 -0.28。

In [ ]:
import numpy as np, mujoco

Z0 = -0.28   # 站立腳掌深度（task3 值），amp=0 時的站姿

def compute_ik_constants(xml):
    m = mujoco.MjModel.from_xml_path(xml)
    d = mujoco.MjData(m)
    fl_geom  = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, "FL")
    fl_thigh = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, "FL_thigh")
    def foot_xz(th, ca):
        mujoco.mj_resetDataKeyframe(m, d, 0)   # home keyframe
        d.qpos[8] = th; d.qpos[9] = ca         # FL thigh, calf
        mujoco.mj_forward(m, d)
        p = d.geom_xpos[fl_geom] - d.xpos[fl_thigh]
        return np.array([p[0], p[2]])
    f0 = foot_xz(0.9, -1.8); e = 1e-3
    J = np.zeros((2, 2))
    J[:, 0] = (foot_xz(0.9 + e, -1.8) - foot_xz(0.9 - e, -1.8)) / (2 * e)
    J[:, 1] = (foot_xz(0.9, -1.8 + e) - foot_xz(0.9, -1.8 - e)) / (2 * e)
    jinv = np.linalg.inv(J)
    center = np.array([f0[0], Z0])
    return f0.astype(np.float32), jinv.astype(np.float32), center.astype(np.float32)

F0_np, JINV_np, CENTER_np = compute_ik_constants(SCENE)
print("F0:", F0_np, "CENTER:", CENTER_np)
print("JINV:\n", JINV_np)

## 第 4 步：MJX 環境（自足版，不依賴 Playground 內部 API）

一個實作 brax `Env` 介面的環境：
- **PD 對齊 task3**：`apply_pd()` 把位置伺服增益改成 **kp=90 / kd=3、力矩上限 ±23.7/膝±45.43**，與你 task3 的軟體 PD 完全等效（位置伺服 `gainprm=[kp,0,0],biasprm=[0,-kp,-kd]` 數學上就是 `kp*(ctrl-q)-kd*q̇`）。之後接回 `Go2Gait` 底層無落差。
- **動作 5 維** → CPG → 12 個關節目標角度 → 直接當 `ctrl`（位置伺服）→ MJX 走 5 個物理子步（控制 50Hz）。
- **觀測 51 維**：重力方向(3)+身體線速度(3)+角速度(3)+關節角相對home(12)+關節速度(12)+指令(3)+上一動作(5)+CPG狀態 r(4),r_dot(4),sin/cos(phi)(2)。
- **獎勵**：追指令線速度 + 追 yaw 角速度 − 傾斜 − 高度偏 − 動作抖動 + 存活。
- **終止**：身體過低或翻覆。
- CPG 狀態存在 `state.info`（JAX 環境不能用物件屬性存每步狀態）。

In [ ]:
import functools
import jax
from brax.envs.base import Env, State
from mujoco import mjx

CTRL_DT = 0.02
SIM_DT  = 0.004
N_FRAMES = int(round(CTRL_DT / SIM_DT))
HOME12 = jnp.array([0.0, 0.9, -1.8] * 4)
KP_NOM, KD_NOM = 90.0, 3.0        # 對齊 task3 的軟體 PD 增益
KNEE_IDX = [2, 5, 8, 11]          # 四個膝(calf)致動器索引


def apply_pd(m, kp=KP_NOM, kd=KD_NOM):
    # 把位置伺服致動器改成「等效於 task3 軟體 PD」：force = kp*(ctrl-q) - kd*qdot
    # 位置伺服 gainprm=[kp,0,0], biasprm=[0,-kp,-kd] 在數學上就是這條 PD。
    m.actuator_gainprm[:, 0] = kp
    m.actuator_biasprm[:, 0] = 0.0
    m.actuator_biasprm[:, 1] = -kp
    m.actuator_biasprm[:, 2] = -kd
    fr = np.full(m.nu, 23.7); fr[KNEE_IDX] = 45.43     # 對齊 task3 go2.xml 力矩上限
    m.actuator_forcerange[:, 0] = -fr
    m.actuator_forcerange[:, 1] = fr
    m.actuator_forcelimited[:] = 1
    return m


def _quat_inv(q):
    return jnp.array([q[0], -q[1], -q[2], -q[3]])

def _quat_rot(q, v):        # rotate v by q (body-vec -> world)
    u = q[1:4]
    t = 2.0 * jnp.cross(u, v)
    return v + q[0] * t + jnp.cross(u, t)

def _world_to_body(quat, v):
    return _quat_rot(_quat_inv(quat), v)


class Go2CpgEnv(Env):
    def __init__(self, f0, jinv, center):
        m = mujoco.MjModel.from_xml_path(SCENE)
        m.opt.timestep = SIM_DT
        m = apply_pd(m)                    # ★ PD 對齊 task3（kp=90/kd=3）
        self._mj_model = m
        self.sys = mjx.put_model(m)        # ★ 命名為 sys，domain randomization 會替換它
        self._init_q = jnp.array(m.key_qpos[0])
        self._lo = jnp.array(m.actuator_ctrlrange[:, 0])
        self._hi = jnp.array(m.actuator_ctrlrange[:, 1])
        self._f0 = jnp.array(f0)
        self._jinv = jnp.array(jinv)
        self._center = jnp.array(center)

    @property
    def observation_size(self):
        return 51

    @property
    def action_size(self):
        return 5

    @property
    def backend(self):
        return "mjx"

    def _sample_cmd(self, rng):
        k1, k2, k3 = jax.random.split(rng, 3)
        vx = jax.random.uniform(k1, (), minval=0.0, maxval=1.0)
        vy = jax.random.uniform(k2, (), minval=-0.3, maxval=0.3)
        wz = jax.random.uniform(k3, (), minval=-1.0, maxval=1.0)
        return jnp.array([vx, vy, wz])

    def _base(self, data):
        quat = data.qpos[3:7]
        gyro = data.qvel[3:6]                          # 自由關節角速度=本體系
        blin = _world_to_body(quat, data.qvel[0:3])    # 線速度 世界->本體
        grav = _world_to_body(quat, jnp.array([0.0, 0.0, -1.0]))
        return quat, gyro, blin, grav

    def _obs(self, data, info):
        _, gyro, blin, grav = self._base(data)
        jpos = data.qpos[7:19] - HOME12
        jvel = data.qvel[6:18]
        cpg = info["cpg"]
        return jnp.concatenate([
            grav, blin, gyro, jpos, jvel,
            info["cmd"], info["last_action"],
            cpg["r"], cpg["r_dot"],
            jnp.array([jnp.sin(cpg["phi"]), jnp.cos(cpg["phi"])]),
        ])

    def reset(self, rng):
        rng, crng = jax.random.split(rng)
        data = mjx.make_data(self.sys).replace(qpos=self._init_q)
        data = mjx.forward(self.sys, data)
        info = {"rng": rng, "cmd": self._sample_cmd(crng),
                "cpg": cpg_init(), "last_action": jnp.zeros(5)}
        obs = self._obs(data, info)
        metrics = {"reward": jnp.zeros(()), "r_lin": jnp.zeros(()),
                   "r_yaw": jnp.zeros(()), "height": jnp.zeros(())}
        return State(data, obs, jnp.zeros(()), jnp.zeros(()), metrics, info)

    def step(self, state, action):
        mu, omega = action_to_cpg_cmd(action)
        cpg = cpg_step(state.info["cpg"], mu, omega, CTRL_DT)
        q_des = cpg_to_joint_targets(cpg, self._f0, self._jinv, self._center)
        ctrl = jnp.clip(q_des, self._lo, self._hi)

        def one(d, _):
            return mjx.step(self.sys, d.replace(ctrl=ctrl)), None
        data, _ = jax.lax.scan(one, state.pipeline_state, None, N_FRAMES)

        info = {**state.info, "cpg": cpg, "last_action": action}
        obs = self._obs(data, info)

        _, gyro, blin, grav = self._base(data)
        cmd = info["cmd"]
        r_lin = jnp.exp(-((blin[0] - cmd[0]) ** 2 + (blin[1] - cmd[1]) ** 2) / 0.25)
        r_yaw = jnp.exp(-((gyro[2] - cmd[2]) ** 2) / 0.25)
        upright_pen = grav[0] ** 2 + grav[1] ** 2
        height = data.qpos[2]
        height_pen = (height - 0.30) ** 2
        act_rate = jnp.sum((action - state.info["last_action"]) ** 2)
        reward = (1.5 * r_lin + 0.8 * r_yaw
                  - 1.0 * upright_pen - 0.5 * height_pen
                  - 0.1 * act_rate + 0.05)
        done = jnp.where((height < 0.18) | (grav[2] > -0.4), 1.0, 0.0)
        metrics = {"reward": reward, "r_lin": r_lin, "r_yaw": r_yaw, "height": height}
        return state.replace(pipeline_state=data, obs=obs, reward=reward, done=done,
                             metrics=metrics, info=info)

print("env defined")

## 第 4.5 步：Domain Randomization（抗擾 + 縮小 sim-to-sim/real 落差）

每個平行環境用**稍微不同的物理參數**訓練，policy 就學會不依賴某組精確數值。這裡隨機化三類：
- **地面摩擦**（0.5~1.0）
- **PD 增益 kp/kd**（kp 75~105、kd 2~4，繞著 task3 的 90/3）→ 直接讓 policy 對「底層 PD 到底多硬」不敏感，正好保護你 task3 vs 訓練的增益差。
- **各連桿質量**（±10%）

Brax 透過 `randomization_fn` 支援：它回傳「沿環境軸批次化的模型」+ `in_axes`。函數會被 `ppo.train` 自動對 `num_envs` 個 rng 呼叫。

In [ ]:
def domain_randomize(sys, rng):
    @jax.vmap
    def per_env(rng):
        k1, k2, k3, k4 = jax.random.split(rng, 4)
        friction = jax.random.uniform(k1, minval=0.5, maxval=1.0)
        geom_friction = sys.geom_friction.at[:, 0].set(friction)
        kp = jax.random.uniform(k2, minval=75.0, maxval=105.0)
        kd = jax.random.uniform(k3, minval=2.0, maxval=4.0)
        gain = sys.actuator_gainprm.at[:, 0].set(kp)
        bias = sys.actuator_biasprm.at[:, 1].set(-kp).at[:, 2].set(-kd)
        dmass = jax.random.uniform(k4, (sys.nbody,), minval=0.9, maxval=1.1)
        body_mass = sys.body_mass * dmass
        return geom_friction, gain, bias, body_mass

    geom_friction, gain, bias, body_mass = per_env(rng)
    in_axes = jax.tree_util.tree_map(lambda x: None, sys)
    in_axes = in_axes.replace(geom_friction=0, actuator_gainprm=0,
                              actuator_biasprm=0, body_mass=0)
    sys = sys.replace(geom_friction=geom_friction, actuator_gainprm=gain,
                      actuator_biasprm=bias, body_mass=body_mass)
    return sys, in_axes

print("domain_randomize ready")

## Smoke test（開訓練前必跑！便宜地抓錯）

jit 一次 reset/step，確認形狀正確、不報錯。這裡若過，後面訓練九成會動；這裡若掛，多半是 MJX/brax 版本 API 差異，照錯誤訊息微調本格上方的環境即可，別急著開訓練。

In [ ]:
env = Go2CpgEnv(F0_np, JINV_np, CENTER_np)
rng = jax.random.PRNGKey(0)
state = jax.jit(env.reset)(rng)
print("obs shape:", state.obs.shape, "(應為 (51,))")
state = jax.jit(env.step)(state, jnp.zeros(5))
print("after step -> reward:", float(state.reward), "done:", float(state.done),
      "height:", float(state.metrics["height"]))
print("SMOKE TEST PASSED" if state.obs.shape == (51,) else "CHECK OBS SIZE")

## 第 5 步：Brax PPO 訓練（GPU）

在 T4 上約數分鐘~十幾分鐘。`eval/episode_reward` 應隨步數上升。
- 記憶體不足（OOM）就把 `num_envs` 降到 1024。
- 學不動就先把指令固定成直走（把 `_sample_cmd` 的 vy、wz 範圍設 0）、或加大 `num_timesteps`。

In [ ]:
import functools, time
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

env = Go2CpgEnv(F0_np, JINV_np, CENTER_np)

network_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=(128, 128, 128),
    value_hidden_layer_sizes=(256, 256, 256),
)

train_fn = functools.partial(
    ppo.train,
    num_timesteps=50_000_000,
    num_evals=10,
    episode_length=1000,
    num_envs=2048,
    batch_size=256,
    num_minibatches=32,
    unroll_length=20,
    num_updates_per_batch=4,
    learning_rate=3e-4,
    entropy_cost=1e-2,
    discounting=0.97,
    normalize_observations=True,
    network_factory=network_factory,
    randomization_fn=domain_randomize,   # ★ 開啟 domain randomization
    seed=0,
)

_t0 = time.time()
rewards = []
def progress(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0))
    rewards.append((step, r))
    print(f"step {step:>10,}  reward {r:8.2f}  ({time.time()-_t0:.0f}s)")

make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)
print("training done")

In [ ]:
import matplotlib.pyplot as plt
xs = [s for s, _ in rewards]; ys = [r for _, r in rewards]
plt.plot(xs, ys, marker="o"); plt.xlabel("env steps"); plt.ylabel("eval reward")
plt.title("CPG-RL training"); plt.grid(True); plt.show()

## 看成果：CPU rollout 存影片

用訓好的策略在一般 MuJoCo（CPU）跑一段並錄影。**這格的 obs 組法就是你本機推論要用的樣板**——欄位順序/單位/控制頻率必須和上面環境完全一致（sim-to-sim）。

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
import numpy as np, mujoco, mediapy as media

infer = jax.jit(make_inference_fn(params, deterministic=True))

# --- NumPy 版 CPG（與 JAX 版邏輯逐行對應）---
def cpg_init_np():
    return {"r": np.zeros(4), "r_dot": np.zeros(4), "phi": 0.0}

def cpg_step_np(c, mu, omega, dt):
    r, rd, phi = c["r"].copy(), c["r_dot"].copy(), c["phi"]
    h = dt / N_CPG_SUB
    for _ in range(N_CPG_SUB):
        rdd = A_CONV * (A_CONV / 4.0 * (mu - r) - rd)
        rd = rd + rdd * h; r = r + rd * h; phi = phi + 2 * np.pi * omega * h
    return {"r": r, "r_dot": rd, "phi": phi % (2 * np.pi)}

def action_to_cmd_np(a):
    a = np.tanh(a)
    mu = (a[:4] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    omega = (a[4] + 1) / 2 * (OMEGA_MAX - OMEGA_MIN) + OMEGA_MIN
    return mu, omega

def targets_np(c, f0, jinv, center):
    th = c["phi"] + np.array([0, np.pi, np.pi, 0.0]); amp = np.clip(c["r"], 0, 1)
    dx = -STRIDE * amp * np.cos(th); dz = np.maximum(0.0, LIFT * amp * np.sin(th))
    foot = np.stack([center[0] + dx, center[1] + dz], -1)
    dq = (foot - f0) @ jinv.T
    q = np.zeros((4, 3)); q[:, 1] = 0.9 + dq[:, 0]; q[:, 2] = -1.8 + dq[:, 1]
    return q.reshape(12)

def qinv(q): return np.array([q[0], -q[1], -q[2], -q[3]])
def qrot(q, v):
    u = q[1:4]; t = 2 * np.cross(u, v); return v + q[0] * t + np.cross(u, t)
def w2b(q, v): return qrot(qinv(q), v)

def build_obs_np(d, c, cmd, last_a):
    quat = d.qpos[3:7].copy(); gyro = d.qvel[3:6].copy()
    blin = w2b(quat, d.qvel[0:3]); grav = w2b(quat, np.array([0, 0, -1.0]))
    jpos = d.qpos[7:19] - np.array([0, 0.9, -1.8] * 4); jvel = d.qvel[6:18]
    return np.concatenate([grav, blin, gyro, jpos, jvel, cmd, last_a,
                           c["r"], c["r_dot"], [np.sin(c["phi"]), np.cos(c["phi"])]]).astype(np.float32)

m = mujoco.MjModel.from_xml_path(SCENE); m.opt.timestep = SIM_DT
m = apply_pd(m)                         # ★ 預覽也用對齊後的 PD（kp=90/kd=3）
d = mujoco.MjData(m); mujoco.mj_resetDataKeyframe(m, d, 0)
lo = m.actuator_ctrlrange[:, 0]; hi = m.actuator_ctrlrange[:, 1]
ren = mujoco.Renderer(m, 480, 640); cam = mujoco.MjvCamera(); mujoco.mjv_defaultFreeCamera(m, cam)

cmd = np.array([0.6, 0.0, 0.0])            # 直走 0.6 m/s（可改）
c = cpg_init_np(); last_a = np.zeros(5); frames = []
rng = jax.random.PRNGKey(0)
for i in range(500):                        # 500*0.02 = 10s
    obs = build_obs_np(d, c, cmd, last_a)
    act, _ = infer(jnp.array(obs), rng); act = np.array(act)
    mu, omega = action_to_cmd_np(act); c = cpg_step_np(c, mu, omega, CTRL_DT)
    q_des = targets_np(c, F0_np, JINV_np, CENTER_np)
    d.ctrl[:] = np.clip(q_des, lo, hi)
    for _ in range(N_FRAMES): mujoco.mj_step(m, d)
    last_a = act
    if i % 2 == 0:
        cam.lookat[:] = d.qpos[:3]; cam.distance = 2.0; cam.elevation = -20
        ren.update_scene(d, cam); frames.append(ren.render())
print("final x:", round(float(d.qpos[0]), 2), "m  height:", round(float(d.qpos[2]), 2))
media.show_video(frames, fps=25)

## 第 6 步：存權重並下載

存下來帶回本機。之後本機推論用**同樣的 `make_ppo_networks` 設定**重建網路 + `model.load_params` 載入即可（見指南第 7 步）。

In [ ]:
from brax.io import model
model.save_params("cpg_rl_params.pkl", params)
print("saved cpg_rl_params.pkl")
try:
    from google.colab import files
    files.download("cpg_rl_params.pkl")
except Exception as e:
    print("手動下載：左側檔案面板右鍵 cpg_rl_params.pkl → Download。", e)

## 帶回本機後

1. `pip install jax brax mujoco`（本機預設 CPU 版即可跑推論）。
2. 用**和本 notebook 完全相同**的 `network_factory` 重建 `make_inference_fn`，`model.load_params("cpg_rl_params.pkl")` 載入。
3. 把上面「CPU rollout」那格的迴圈搬進你 task3 的架構，物理用你本機的 `mujoco`，策略用 `infer(obs)`。
4. obs 組法（`build_obs_np`）務必逐維對齊，否則策略會亂走。

延伸：動作 5→12 維（每腿獨立頻率+側向）、加腿間耦合、加地形高度觀測、加重 domain randomization 為 sim-to-real 鋪路。